# Retrieval metrics

This notebook calculates nDCG@5 from a constructed ranking and shared relevance
pool, following **Appendix A.3, Table 21** of the thesis. Each rank's contribution
is visible and checked against the reusable Python function. Further cases examine
complete evidence sets, missing positions and validity for aggregation.

**Before running:** follow the setup in the [README](../README.md), then use
*Restart Kernel and Run All Cells*. No private data or model service is needed.

In [1]:
from math import isclose, log2

from IPython.display import Markdown, display
from tabulate import tabulate

from rag_comparison.config import TOP_K
from rag_comparison.metrics import ndcg_at_k, score_question

## 1. Read the inputs

An **Evidence Unit** identifies a document and physical page within a fixed corpus
snapshot. The worked example uses synthetic IDs. A **qrel** is an accepted relevance judgement
for a question and an Evidence Unit (see thesis Section 2.4):

| Grade | Meaning |
|---:|---|
| 0 | Does not contribute to answering the question |
| 1 | Related to the topic, but does not directly support a required fact |
| 2 | Directly supports at least one required fact |

A required fact is a fact that a complete answer must contain. Grade 2 does not
mean that one unit contains every fact needed for the whole answer.

`qrels` is the **shared pool**, not just this system's retrieved units. Both systems
use the same pool within a comparison. Unit F has grade 1 in that pool, even though
it is absent from the returned ranking. B and E have valid grade-0 judgements.

In [2]:
qrels = {"A": 2, "B": 0, "C": 1, "D": 2, "E": 0, "F": 1}
ranked_units = ["A", "B", "C", "D", "E"]

print(f"Evaluation depth: {TOP_K}")
print(f"Shared relevance pool: {qrels}")
print(f"Returned units: {ranked_units}")

Evaluation depth: 5
Shared relevance pool: {'A': 2, 'B': 0, 'C': 1, 'D': 2, 'E': 0, 'F': 1}
Returned units: ['A', 'B', 'C', 'D', 'E']


## 2. Calculate the retrieved ranking's gain

**Discounted cumulative gain (DCG)** rewards relevant evidence near the top.
For rank $i$, the weight is $1 / \log_2(i + 1)$: rank 1 gets full weight, while
later ranks get smaller weights.

The calculation uses the grades **directly as linear gains**:

$$DCG@5 = \sum_{i=1}^{5} \frac{g_i}{\log_2(i + 1)}.$$

Here all five returned units are distinct and judged, so their gains are simply
their qrel grades. Keep full precision in the calculation.

In [3]:
retrieved_gains = [qrels[unit] for unit in ranked_units]
rank_weights = [1 / log2(rank + 1) for rank in range(1, TOP_K + 1)]
retrieved_contributions = [
    gain * weight for gain, weight in zip(retrieved_gains, rank_weights)
]
retrieved_dcg = sum(retrieved_contributions)

print(f"Retrieved gains: {retrieved_gains}")
print(f"DCG@{TOP_K}: {retrieved_dcg:.12f}")

Retrieved gains: [2, 0, 1, 2, 0]
DCG@5: 3.361353116147


## 3. Build the ideal ranking from the shared pool

**Ideal DCG (IDCG)** is the best discounted gain available in this question's
shared pool. Sort the positive grades of distinct pool units in descending order
and take at most five. The pool has four positive units: A, D, C and F.

The ideal gains are therefore `[2, 2, 1, 1, 0]`. The final zero is padding for an
unused ideal position. It does not add a new judgement to the pool.

Including F matters: using only the retrieved grades would give the wrong
normalising denominator. PRIMARY and FINAL comparisons have separate pool versions;
scores from those different bases must not be mixed.

In [4]:
positive_pool_gains = sorted(
    (grade for grade in qrels.values() if grade > 0), reverse=True
)
ideal_gains = (positive_pool_gains + [0] * TOP_K)[:TOP_K]
ideal_contributions = [gain * weight for gain, weight in zip(ideal_gains, rank_weights)]
ideal_dcg = sum(ideal_contributions)

print(f"Ideal gains: {ideal_gains}")
print(f"IDCG@{TOP_K}: {ideal_dcg:.12f}")

Ideal gains: [2, 2, 1, 1, 0]
IDCG@5: 4.192536065216


## 4. Inspect the rank contributions

See thesis Table 21 for the worked calculation. Displayed values are
rounded to six decimal places; the sums still use the unrounded values above.

In [5]:
rank_rows = list(
    zip(
        range(1, TOP_K + 1),
        ranked_units,
        retrieved_gains,
        rank_weights,
        retrieved_contributions,
        ideal_gains,
        ideal_contributions,
    )
)
rank_rows.append(("Sum", None, None, None, retrieved_dcg, None, ideal_dcg))
headers = [
    "Rank",
    "Unit",
    "Gain",
    "Weight",
    "DCG contribution",
    "Ideal gain",
    "IDCG contribution",
]
display(
    Markdown(tabulate(rank_rows, headers=headers, tablefmt="github", floatfmt=".6f"))
)

| Rank   | Unit   |   Gain |   Weight |   DCG contribution |   Ideal gain |   IDCG contribution |
|--------|--------|--------|----------|--------------------|--------------|---------------------|
| 1      | A      |      2 | 1.000000 |           2.000000 |            2 |            2.000000 |
| 2      | B      |      0 | 0.630930 |           0.000000 |            2 |            1.261860 |
| 3      | C      |      1 | 0.500000 |           0.500000 |            1 |            0.500000 |
| 4      | D      |      2 | 0.430677 |           0.861353 |            1 |            0.430677 |
| 5      | E      |      0 | 0.386853 |           0.000000 |            0 |            0.000000 |
| Sum    |        |        |          |           3.361353 |              |            4.192536 |

## 5. Normalise and call the reusable function

**Normalised DCG (nDCG)** compares the observed gain with the ideal gain:

$$nDCG@5 = \frac{DCG@5}{IDCG@5}.$$

The visible calculation explains the formula. The imported function is the
reusable implementation in [`metrics.py`](../src/rag_comparison/metrics.py).
It applies the full rank, duplicate and judgement contract. The function returns
the score rounded to six decimal places.

In [6]:
manual_ndcg = retrieved_dcg / ideal_dcg
score = ndcg_at_k(ranked_units, qrels)

assert isclose(retrieved_dcg, 3.3613531161467862, abs_tol=1e-12)
assert isclose(ideal_dcg, 4.192536065216308, abs_tol=1e-12)
assert score == round(manual_ndcg, 6) == 0.801747

print(f"Manual nDCG before rounding: {manual_ndcg:.12f}")
print(f"Function result: {score:.6f}")

Manual nDCG before rounding: 0.801746976975
Function result: 0.801747


## 6. Interpret the result

The ranking reaches approximately **80.17% of the ideal discounted gain** for
this assumed pool. A score of 1 would match the ideal gain. A score of 0 can be a
valid result when every returned unit has a judged grade of 0 and the pool contains
positive evidence elsewhere.

This percentage is **not the proportion of correct answers**. It measures ranked
source relevance for one question. It also does not prove that all required facts
were retrieved. Evidence Recall@5 measures complete minimal evidence sets separately;
Appendix A.3 does not specify such a set.

All five positions in this example are judged, so Judged@5 is 5/5 = 1.0.
This remains a hypothetical worked example, not a measured system result.

## 7. Preserve duplicates and invalid cases

| Situation | Function behaviour |
|---|---|
| A repeated evidence ID | Keeps its rank position; only its first occurrence gains credit |
| Valid grade 0 | Contributes zero gain and remains a judged position |
| Fewer than five positions | Returns `None` |
| Failed projection, represented by `None` | Returns `None` |
| Evidence ID absent from qrels | Returns `None`, not an artificial grade-0 judgement |
| No positive ideal gain | Returns `None`; normalisation has no valid denominator |
| Grade outside 0/1/2 anywhere in the pool | Raises `ValueError` |

The function returns a valid nDCG value or `None`. Valid aggregation also requires
separate coverage and failure information. The score alone does not provide these
fields; a partially computable score must never be treated as a valid observation.

In [7]:
examples = [
    ("Duplicate keeps rank", ndcg_at_k(["A", "A", "D", "C", "F"], qrels)),
    ("Judged zeros", ndcg_at_k(["B", "E", "B", "E", "B"], qrels)),
    ("Missing rank", ndcg_at_k(["A", "B", "C", "D"], qrels)),
    ("Failed projection", ndcg_at_k(["A", "B", None, "D", "E"], qrels)),
    ("Unjudged evidence", ndcg_at_k(["A", "B", "C", "D", "unjudged"], qrels)),
    ("No positive pool gain", ndcg_at_k(ranked_units, dict.fromkeys(qrels, 0))),
]
for label, result in examples:
    print(f"{label:24} {result}")

assert [result for _, result in examples] == [0.910554, 0.0, None, None, None, None]

Duplicate keeps rank     0.910554
Judged zeros             0.0
Missing rank             None
Failed projection        None
Unjudged evidence        None
No positive pool gain    None


## 8. Check complete evidence sets and judgement coverage

A **minimal evidence set** lists the source units jointly sufficient for an answer.
A question can have several alternative sets. **Evidence Recall@5 is binary** (Equation (3)): it is
1 if at least one complete set is present, and 0 otherwise. It is not the fraction
of required units retrieved. **Partial overlap** records that fraction separately
as a diagnostic.

**Judged@5** (Equation (4)) counts positions with an accepted qrel and divides by five. A judged
zero and a duplicate count as judged; an unknown or unprojectable unit does not.

The following reference sets are hypothetical. They are separate from Appendix A.3,
which does not specify minimal evidence sets.


In [8]:
example_qrels = {"X1": 2, "X2": 2, "X3": 2, "X4": 1, "X5": 0, "X6": 0}
example_sets = [{"X1", "X2"}, {"X3", "X4"}]
example_ranking = ["X1", "X3", "X5", "X6", "X5"]
question_score = score_question(example_ranking, example_qrels, example_sets)

display(
    Markdown(
        tabulate(
            list(question_score.items()),
            headers=["Score field", "Value"],
            tablefmt="github",
        )
    )
)

| Score field              | Value             |
|--------------------------|-------------------|
| ndcg_at_k                | 0.695117          |
| dcg                      | 3.261859507142915 |
| idcg                     | 4.692536065216308 |
| judged_at_k              | 1.0               |
| evidence_recall_at_k     | 0.0               |
| partial_overlap_at_k     | 0.5               |
| ranked_position_count    | 5                 |
| missing_position_count   | 0                 |
| projection_failure_count | 0                 |
| unjudged_unit_ids        | []                |
| metric_validity_status   | fully_judged      |

## 9. Keep valid outcomes separate from diagnostic calculations

A ranking enters paired retrieval analysis only when each of its five positions
maps unambiguously to a judged Evidence Unit and the shared pool has positive ideal gain:
`metric_validity_status == "fully_judged"` **and** `idcg > 0`.
Both nDCG@5 and Evidence Recall@5 use that same valid retrieval basis.

`score_question` exposes its calculations and failure counts. Its nDCG field can
still contain a diagnostic value when some positions are missing. Such a value
is not a valid observation and must not become an artificial zero. The simpler
`ndcg_at_k` interface already returns `None` for these invalid rankings.


In [9]:
valid_retrieval = (
    question_score["metric_validity_status"] == "fully_judged"
    and question_score["idcg"] > 0
)
retrieval_outcomes = {
    "nDCG@5": question_score["ndcg_at_k"] if valid_retrieval else None,
    "Evidence Recall@5": (
        question_score["evidence_recall_at_k"] if valid_retrieval else None
    ),
}

assert valid_retrieval
assert question_score["judged_at_k"] == 1.0
assert question_score["evidence_recall_at_k"] == 0.0
assert question_score["partial_overlap_at_k"] == 0.5
print(f"Valid retrieval outcomes: {retrieval_outcomes}")
print("Half of an alternative evidence set is present; neither set is complete.")

Valid retrieval outcomes: {'nDCG@5': 0.695117, 'Evidence Recall@5': 0.0}
Half of an alternative evidence set is present; neither set is complete.


## 10. Inspect missing and repeated positions

In the first three cases below, the available evidence still gives a positive
**diagnostic** nDCG. The valid-score interface returns `None`, and these questions
do not enter retrieval aggregates. Missing ranks are not removed from the Judged@5
denominator, and an unjudged unit is not assigned a grade of zero.

Repeated units keep their original positions and only their first occurrence
contributes gain. The final case is a valid zero: every returned position has an
accepted grade of zero, while positive evidence exists elsewhere in the shared pool.


In [10]:
rank_cases = [
    ("Missing rank", ["X1", "X3", "X5", "X6"]),
    ("Failed projection", ["X1", None, "X5", "X6", "X5"]),
    ("Unjudged unit", ["X1", "unjudged", "X5", "X6", "X5"]),
    ("Repeated positive unit", ["X1", "X1", "X2", "X3", "X5"]),
    ("Judged zeros", ["X5", "X6", "X5", "X6", "X5"]),
]
status_rows = []
for label, ranking in rank_cases:
    result = score_question(ranking, example_qrels, example_sets)
    status_rows.append(
        (
            label,
            result["metric_validity_status"],
            result["judged_at_k"],
            result["ndcg_at_k"],
            ndcg_at_k(ranking, example_qrels),
        )
    )
display(
    Markdown(
        tabulate(
            status_rows,
            headers=[
                "Case",
                "Validity status",
                "Judged@5",
                "Diagnostic nDCG",
                "Valid nDCG",
            ],
            tablefmt="github",
            floatfmt=".6f",
        )
    )
)
assert all(row[-1] is None and row[-2] > 0 for row in status_rows[:3])
assert status_rows[-1][-1] == 0.0

| Case                   | Validity status             |   Judged@5 |   Diagnostic nDCG |   Valid nDCG |
|------------------------|-----------------------------|------------|-------------------|--------------|
| Missing rank           | missing_ranked_positions    |   0.800000 |          0.695117 |              |
| Failed projection      | projection_failures_present |   0.800000 |          0.426209 |              |
| Unjudged unit          | unjudged_positions_present  |   0.800000 |          0.426209 |              |
| Repeated positive unit | fully_judged                |   1.000000 |          0.822871 |     0.822871 |
| Judged zeros           | fully_judged                |   1.000000 |          0.000000 |     0.000000 |

## Saved RFC 2795 retrievals

The following calculation uses the same `score_question` function with actual stored
Classical-RAG retrievals and illustrative source-reviewed reference grades. Each answerable
question has a grade for every page in the 20-page example corpus. The shared pool
is fixed for this example; these values are separate from the private experiment results.

Complete reference evidence means that all units of at least one listed minimal
evidence set are present in the five retrieved positions. The timeout question
has no answerable reference set and is outside the primary retrieval assessment.


In [11]:
import hashlib
import json
from pathlib import Path

example_dir = Path("../examples")
rfc_example = json.loads(
    (example_dir / "data/reference_packages.json").read_text(encoding="utf-8")
)
run_dir = example_dir / "classical_rag"
run_manifest = json.loads((run_dir / "run_manifest.json").read_text(encoding="utf-8"))
assert run_manifest["question_set_version"] == rfc_example["question_set_version"]
retrieval_path = run_dir / run_manifest["artifacts"]["retrieval"]["path"]
assert (
    hashlib.sha256(retrieval_path.read_bytes()).hexdigest()
    == run_manifest["artifacts"]["retrieval"]["sha256"]
)
retrievals = [
    json.loads(line)
    for line in retrieval_path.read_text(encoding="utf-8").splitlines()
    if line
]
references = {
    q["question_id"]: q["reference_package"] for q in rfc_example["questions"]
}

In [12]:
rfc_rows = []
for record in retrievals:
    ref = references[record["question_id"]]
    ranked_ids = [hit["evidence_unit_id"] for hit in record["hits"]]
    sets = [set(group) for group in ref["minimal_evidence_sets"]]
    result = score_question(ranked_ids, ref["qrels"], sets)
    valid = result["metric_validity_status"] == "fully_judged" and result["idcg"] > 0
    score = result["ndcg_at_k"] if valid else None
    complete = result["evidence_recall_at_k"] if valid else None
    if ref["abstention_expected"]:
        score_display = evidence_display = "not applicable"
    else:
        score_display = score if score is not None else "invalid"
        evidence_display = complete if complete is not None else "invalid"
    rfc_rows.append(
        (
            record["question_id"],
            [hit["page"] for hit in record["hits"]],
            score_display,
            evidence_display,
        )
    )
display(
    Markdown(
        tabulate(
            rfc_rows,
            headers=[
                "Question",
                "Retrieved pages",
                "nDCG@5",
                "Complete reference evidence",
            ],
            tablefmt="github",
        )
    )
)
assert all(len(record["hits"]) == 5 for record in retrievals)

| Question        | Retrieved pages      | nDCG@5         | Complete reference evidence   |
|-----------------|----------------------|----------------|-------------------------------|
| DEMO-IMPS-Q-001 | [8, 7, 9, 17, 10]    | 0.825402       | 0.0                           |
| DEMO-IMPS-Q-002 | [12, 13, 3, 16, 14]  | 0.85163        | 1.0                           |
| DEMO-IMPS-Q-003 | [15, 13, 17, 16, 12] | 0.902024       | 1.0                           |
| DEMO-IMPS-Q-004 | [9, 10, 11, 17, 8]   | not applicable | not applicable                |

## Function interface and references

- Import `ndcg_at_k` when you need the valid score for one ranked question.
- Import `score_question` for judgement coverage, failure status and evidence-set metrics.
- Read [`config.py`](../src/rag_comparison/config.py) for the shared evaluation depth.
- See [the focused tests](../tests/test_metrics.py) for the thesis example and
  scientific edge cases. Run them from the repository folder with
  `uv run --extra pdf --extra graphrag --extra evaluation python -m unittest discover -s tests -p test_metrics.py -v`.

**Further reading:** thesis Section 2.4 defines relevance grades; Section 2.6,
Equations (1)–(4), defines the metrics. Appendix A.3, Table 21, gives the worked calculation.